# Detección de contornos

**Duración estimada:** 60 minutos

## Objetivo

En este cuaderno vas a detectar contornos a partir de una máscara sobre globos naranjas. La idea es ver, paso a paso, cómo una imagen color termina convertida en fronteras que después podremos medir.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


CARPETA_IMAGENES = Path("Imagenes")


def abrir_imagen_bgr(nombre_archivo):
    """Abre una imagen en color usando el orden BGR de OpenCV."""
    ruta = CARPETA_IMAGENES / nombre_archivo
    imagen_bgr = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    if imagen_bgr is None:
        raise FileNotFoundError(f"No pude abrir la imagen: {ruta}")
    return imagen_bgr


def abrir_imagen_rgb(nombre_archivo):
    """Abre una imagen y la convierte a RGB para mostrarla con Matplotlib."""
    imagen_bgr = abrir_imagen_bgr(nombre_archivo)
    imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    return imagen_rgb


def preparar_eje_imagen(eje, imagen, titulo, mapa=None):
    """Muestra una imagen conservando ejes y coordenadas visibles."""
    eje.imshow(imagen, cmap=mapa)
    eje.set_title(titulo, loc="left", fontweight="bold")
    eje.set_xlabel("x (columnas)")
    eje.set_ylabel("y (filas)")

    if imagen.ndim == 2:
        alto, ancho = imagen.shape
    else:
        alto, ancho = imagen.shape[:2]

    paso_x = max(50, ancho // 6)
    paso_y = max(50, alto // 6)
    eje.set_xticks(np.arange(0, ancho + 1, paso_x))
    eje.set_yticks(np.arange(0, alto + 1, paso_y))
    eje.grid(color="white", alpha=0.25, linewidth=0.6)


def mostrar_una_imagen(imagen, titulo, mapa=None, tamano=(8, 6)):
    fig, eje = plt.subplots(figsize=tamano, constrained_layout=True)
    preparar_eje_imagen(eje, imagen, titulo, mapa)
    plt.show()


def mostrar_varias_imagenes(imagenes, titulos, mapas=None, tamano=(15, 5)):
    if mapas is None:
        mapas = [None] * len(imagenes)

    fig, ejes = plt.subplots(1, len(imagenes), figsize=tamano, constrained_layout=True)
    if len(imagenes) == 1:
        ejes = [ejes]

    for eje, imagen, titulo, mapa in zip(ejes, imagenes, titulos, mapas):
        preparar_eje_imagen(eje, imagen, titulo, mapa)

    plt.show()


def mostrar_histograma_gris(imagen_gris, titulo):
    histograma, bordes = np.histogram(imagen_gris.ravel(), bins=256, range=(0, 256))
    plt.figure(figsize=(9, 4))
    plt.plot(bordes[:-1], histograma, color="black")
    plt.title(titulo, loc="left", fontweight="bold")
    plt.xlabel("Intensidad")
    plt.ylabel("Cantidad de píxeles")
    plt.grid(alpha=0.3)
    plt.xlim(0, 255)
    plt.show()


def mostrar_histogramas_bgr(imagen_bgr, titulo_general):
    nombres = ["azul", "verde", "rojo"]
    colores = ["tab:blue", "tab:green", "tab:red"]
    fig, ejes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

    for indice in range(3):
        histograma = cv2.calcHist([imagen_bgr], [indice], None, [256], [0, 256]).ravel()
        ejes[indice].plot(histograma, color=colores[indice])
        ejes[indice].set_title(f"Canal {nombres[indice]}", loc="left", fontweight="bold")
        ejes[indice].set_xlabel("Intensidad")
        ejes[indice].set_ylabel("Frecuencia")
        ejes[indice].grid(alpha=0.25)

    fig.suptitle(titulo_general, x=0.01, ha="left", fontweight="bold")
    plt.show()

imagen_bgr = abrir_imagen_bgr("globos.jpg")
imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
imagen_hsv = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2HSV)

mostrar_una_imagen(imagen_rgb, "Imagen de globos con ejes visibles")


## 1. Construir una máscara de naranjas

Si el objetivo es detectar contornos de globos naranjas, primero necesitamos una máscara razonable. En esta versión ajustamos el rango para que apunte mejor a naranjas y no quede corrido hacia amarillos.


In [ ]:
rango_naranja_bajo = np.array([8, 80, 80])
rango_naranja_alto = np.array([22, 255, 255])

mascara_naranja = cv2.inRange(imagen_hsv, rango_naranja_bajo, rango_naranja_alto)

kernel = np.ones((5, 5), np.uint8)
mascara_naranja = cv2.morphologyEx(mascara_naranja, cv2.MORPH_OPEN, kernel, iterations=1)
mascara_naranja = cv2.morphologyEx(mascara_naranja, cv2.MORPH_CLOSE, kernel, iterations=2)

mostrar_varias_imagenes(
    [imagen_rgb, mascara_naranja],
    ["Imagen original", "Máscara de globos naranjas"],
    [None, "gray"],
    tamano=(14, 5),
)


## 2. Detectar contornos sobre la máscara

Ahora sí usamos `cv2.findContours()`. El resultado no es un dibujo terminado: es una lista de bordes. Después podemos decidir cuáles conservar y cómo mostrarlos.


In [ ]:
contornos_encontrados, jerarquia = cv2.findContours(
    mascara_naranja.copy(),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE,
)

contornos_filtrados = []
for contorno in contornos_encontrados:
    area = cv2.contourArea(contorno)
    if area > 500:
        contornos_filtrados.append(contorno)

imagen_con_contornos = imagen_rgb.copy()
cv2.drawContours(imagen_con_contornos, contornos_filtrados, -1, (255, 140, 0), 3)

mostrar_varias_imagenes(
    [mascara_naranja, imagen_con_contornos],
    ["Máscara binaria", "Contornos naranjas detectados"],
    ["gray", None],
    tamano=(14, 5),
)

print("Cantidad de contornos detectados antes del filtrado:", len(contornos_encontrados))
print("Cantidad de contornos conservados:", len(contornos_filtrados))
print("Jerarquía devuelta por OpenCV:", None if jerarquia is None else jerarquia.shape)


In [ ]:
print("Áreas de los primeros contornos conservados:")

areas = []
for contorno in contornos_filtrados:
    areas.append(cv2.contourArea(contorno))

areas.sort(reverse=True)
for area in areas[:8]:
    print(f"- {area:.1f} píxeles cuadrados")


## Cierre

Un contorno depende totalmente de la máscara que le da origen. Si la máscara queda mal, el contorno también. En el próximo cuaderno vamos a volver a un enfoque más controlado para estudiar propiedades geométricas con mayor claridad.
